# Solar System Visualization
**Note:** Matplotlib rendering uses CPU (no GPU option exists). GPU encoding will be used if available.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import re
import subprocess
import os

In [2]:
# Check if NVENC (GPU encoding) is available
def check_nvenc():
    try:
        result = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'], 
                              capture_output=True, text=True)
        return 'h264_nvenc' in result.stdout
    except:
        return False

has_nvenc = check_nvenc()
print(f"GPU encoding (NVENC): {'Available ✓' if has_nvenc else 'Not available - using CPU'}")

GPU encoding (NVENC): Not available - using CPU


In [3]:
# Read config
config = {}
with open('config.txt', 'r') as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#'):
            key, value = line.split('=')
            config[key.strip()] = float(value.strip())

print(f"Video: {config['simulation_time_seconds']}s @ {config['fps']} fps")
print(f"Time step: {config['dt']/86400} days per frame")

Video: 10.0s @ 60.0 fps
Time step: 1.0 days per frame


In [4]:
# Parse CSV
def get_xyz(s):
    nums = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', s)
    return float(nums[0]), float(nums[1]), float(nums[2]) if len(nums) >= 3 else (0,0,0)

def parse_csv(filename):
    frames, names, xs, ys = [], [], [], []
    with open(filename, 'r') as f:
        f.readline()
        for line in f:
            if not line.strip(): continue
            parts = []
            current, depth = "", 0
            for c in line:
                if c == '(': depth += 1
                elif c == ')': depth -= 1
                if c == ',' and depth == 0:
                    parts.append(current)
                    current = ""
                else:
                    current += c
            if current: parts.append(current)
            
            if len(parts) >= 6:
                x, y, z = get_xyz(parts[5])
                frames.append(int(parts[0]))
                names.append(parts[1])
                xs.append(x)
                ys.append(y)
    
    return pd.DataFrame({'frame': frames, 'name': names, 'x': xs, 'y': ys})

df_gpu = parse_csv('output_gpu.csv')
df_cpu = parse_csv('output_cpu.csv')
print(f"Loaded {df_gpu['frame'].max()+1} frames, {len(df_gpu['name'].unique())} objects")

Loaded 600 frames, 5 objects


In [5]:
def make_video(data, output_file, title):
    frames = sorted(data['frame'].unique())
    objects = data['name'].unique()
    colors = {'Sun':'yellow', 'Mercury':'gray', 'Venus':'orange', 
              'Earth':'blue', 'Moon':'lightgray', 'Mars':'red'}
    
    fig, ax = plt.subplots(figsize=(19.2, 10.8))
    m = max(data['x'].abs().max(), data['y'].abs().max()) * 1.1
    ax.set_xlim(-m, m)
    ax.set_ylim(-m, m)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3, linewidth=2)
    ax.set_xlabel('X (meters)', fontsize=20)
    ax.set_ylabel('Y (meters)', fontsize=20)
    ax.set_title(title, fontsize=24)
    
    dots, trails, trail_data = {}, {}, {obj: {'x':[], 'y':[]} for obj in objects}
    
    for obj in objects:
        color = colors.get(obj, 'purple')
        trails[obj], = ax.plot([], [], '-', color=color, alpha=0.4, linewidth=2)
        size = 500 if obj=='Sun' else (80 if obj=='Moon' else 150)
        dots[obj] = ax.scatter([], [], s=size, c=color, label=obj, edgecolors='black', linewidths=1.5)
    
    ax.legend(loc='upper right', fontsize=16)
    info = ax.text(0.02, 0.98, '', transform=ax.transAxes, va='top', fontsize=18,
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    def update(i):
        if i % 30 == 0: print(f"  {i}/{len(frames)}", end='\r')
        frame_data = data[data['frame'] == frames[i]]
        for obj in objects:
            obj_data = frame_data[frame_data['name'] == obj]
            if not obj_data.empty:
                x, y = obj_data['x'].values[0], obj_data['y'].values[0]
                dots[obj].set_offsets([[x, y]])
                trail_data[obj]['x'].append(x)
                trail_data[obj]['y'].append(y)
                trails[obj].set_data(trail_data[obj]['x'], trail_data[obj]['y'])
        days = frames[i] * config['dt'] / 86400
        info.set_text(f'Frame: {frames[i]}\nDays: {days:.1f}')
    
    print(f"\nRendering {len(frames)} frames (CPU)...")
    anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=50)
    
    print(f"Encoding video...")
    fps = int(config['fps'])
    
    # Use GPU encoding if available
    if has_nvenc:
        print("Using GPU encoder (NVENC)")
        writer = animation.FFMpegWriter(
            fps=fps, codec='h264_nvenc', bitrate=5000,
            extra_args=['-preset', 'fast', '-pix_fmt', 'yuv420p'])
    else:
        print("Using CPU encoder")
        writer = animation.FFMpegWriter(
            fps=fps, codec='libx264', bitrate=5000,
            extra_args=['-preset', 'ultrafast', '-pix_fmt', 'yuv420p'])
    
    anim.save(output_file, writer=writer, dpi=100)
    plt.close(fig)
    
    # Force close and flush
    del anim
    del writer
    
    print(f"✓ Saved {output_file}\n")
    return os.path.getsize(output_file) / (1024*1024)

## Generate GPU video (run this first)

In [6]:
size = make_video(df_gpu, 'output_gpu.mp4', 'GPU Simulation')
print(f"GPU video ready! Size: {size:.1f} MB")
print("You can open and watch output_gpu.mp4 now!")


Rendering 600 frames (CPU)...
Encoding video...
Using CPU encoder
✓ Saved output_gpu.mp4

GPU video ready! Size: 2.2 MB
You can open and watch output_gpu.mp4 now!


## Generate CPU video (run this second)

In [ ]:
size = make_video(df_cpu, 'output_cpu.mp4', 'CPU Simulation')
print(f"CPU video ready! Size: {size:.1f} MB")
print("You can open and watch output_cpu.mp4 now!")


Rendering 600 frames (CPU)...
Encoding video...
Using CPU encoder
